In [1]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os

# Konfigurasi path (Sama dengan notebook sebelumnya)
DATA_PATH = '../../../outputs/data-preparation/data_preprocessing_final_no_stem.csv'
INSET_PATH = '../../../../kamus/inset_final.csv'
OUTPUT_DIR = '../../../outputs/sentiment-analysis/RSN'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [2]:
# 2.2 Load Data Preprocessing Final
df = pd.read_csv(DATA_PATH)

print(f"\nData preprocessing berhasil dimuat: {len(df)} tweet")
print(f"Kolom: {df.columns.tolist()}")
df.head()


Data preprocessing berhasil dimuat: 13192 tweet
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertibkan media online DPR pemerintah jangan s...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,kebebasan bersuara berpendapat memang dijamin ...


In [3]:
# 2.3 Load Leksikon InSet dan Definisi Kategori Kata Fungsi
df_inset = pd.read_csv(INSET_PATH)
df_inset['kata'] = df_inset['kata'].astype(str).str.strip().str.lower()

# Definisi kategori kata fungsi (Tetap didefinisikan untuk keperluan analisis/diagnostik)
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}
KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}
PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}
PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL
    | KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
)

print("[INFO] Kategori kata fungsi telah didefinisikan.")

[INFO] Kategori kata fungsi telah didefinisikan.


In [4]:
# 2.4 Mengecek Pengaruh Kata Fungsi di InSet (Tanpa Perlakuan)
# Pada pendekatan ini, ingin melihat seberapa banyak kata fungsi 
# yang akan masuk ke perhitungan skor sentimen secara mentah.

df_found = df_inset[df_inset['kata'].isin(ALL_FUNCTION_WORDS)].copy().sort_values('kata')
total_fw_in_lexicon = len(df_found)

print(f"[DIAGNOSTIK] Total kata fungsi yang TERDAFTAR di InSet: {total_fw_in_lexicon}")
print("\n[KONTEKS] Kata-kata ini akan dianggap bermuatan sentimen (masuk ke matched_words)")
print("karena tidak ada perlakuan 'ignore' maupun 'hapus'.")

if not df_found.empty:
    print("\nContoh kata fungsi yang akan dihitung skornya:")
    print(df_found.head(10).to_string(index=False))
else:
    print("Tidak ada kata fungsi di leksikon.")

[DIAGNOSTIK] Total kata fungsi yang TERDAFTAR di InSet: 22

[KONTEKS] Kata-kata ini akan dianggap bermuatan sentimen (masuk ke matched_words)
karena tidak ada perlakuan 'ignore' maupun 'hapus'.

Contoh kata fungsi yang akan dihitung skornya:
 kata  skor
  aku     2
 anda    -4
 anda    -1
  apa    -3
boleh     2
bukan    -3
 dari    -3
  dia    -3
harus    -5
  itu    -2


In [5]:
# 2.5 Konfigurasi Dictionary dan Tokenisasi
inset_dict = dict(zip(df_inset['kata'], df_inset['skor']))

print(f"\n[KONFIGURASI] Dictionary InSet siap: {len(inset_dict)} entri.")
print("Pendekatan: TANPA IGNORE (Menggunakan skor asli InSet).")

# Fungsi Tokenisasi
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)
print(f"\n[INFO] Tokenisasi selesai. Total token: {df['tokens'].str.len().sum():,}")


[KONFIGURASI] Dictionary InSet siap: 9071 entri.
Pendekatan: TANPA IGNORE (Menggunakan skor asli InSet).

[INFO] Tokenisasi selesai. Total token: 235,559


In [6]:
# 2.6 Fungsi Lexicon Matching (Tanpa Perlakuan)
# Jika kata ada di kamus -> Matched. Jika tidak -> Unmatched.
# Tidak ada filter untuk kata fungsi.

def match_lexicon_no_ignore(tokens, lexicon):
    matched_words = []
    matched_function_words = [] # Kolom tambahan untuk analisis dampak kata fungsi
    unmatched_words = []
    
    for token in tokens:
        token_lower = token.lower()
        if token_lower in lexicon:
            matched_words.append(token)
            # Cek apakah kata yang matched ini sebenarnya adalah kata fungsi
            if token_lower in ALL_FUNCTION_WORDS:
                matched_function_words.append(token)
        else:
            unmatched_words.append(token)
           
    return matched_words, matched_function_words, unmatched_words

print("[INFO] Fungsi matching tanpa ignore siap.")

[INFO] Fungsi matching tanpa ignore siap.


In [7]:
# 2.7 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching (Tanpa Perlakuan Fungsi)...")

df[['matched_words', 'matched_function_words', 'unmatched_words']] = df['tokens'].apply(
    lambda x: pd.Series(match_lexicon_no_ignore(x, inset_dict))
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching (Tanpa Perlakuan Fungsi)...
[INFO] Lexicon matching selesai.


In [8]:
# 2.8 Perhitungan Statistik
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_matched_func = df['matched_function_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()

# Kata fungsi yang dianggap bermuatan sentimen
func_as_sentiment = total_matched_func

print("\n[STATISTIK] Hasil Lexicon Matching (Tanpa Perlakuan):")
print(f"Total kata               : {total_words:,}")
print(f"Matched di InSet         : {total_matched:,} ({(total_matched/total_words)*100:.2f}%)")
print(f"   -> Termasuk Kata Fungsi: {func_as_sentiment:,} ({(func_as_sentiment/total_words)*100:.2f}%)")
print(f"Unmatched                : {total_unmatched:,} ({(total_unmatched/total_words)*100:.2f}%)")
print("\n[ANALISIS] Perhatikan jumlah 'Matched Termasuk Kata Fungsi'.")
print("Angka ini menunjukkan potensi bias sentimen karena kata fungsi dihitung sebagai kata bermuatan.")


[STATISTIK] Hasil Lexicon Matching (Tanpa Perlakuan):
Total kata               : 235,559
Matched di InSet         : 77,315 (32.82%)
   -> Termasuk Kata Fungsi: 12,253 (5.20%)
Unmatched                : 158,244 (67.18%)

[ANALISIS] Perhatikan jumlah 'Matched Termasuk Kata Fungsi'.
Angka ini menunjukkan potensi bias sentimen karena kata fungsi dihitung sebagai kata bermuatan.


In [9]:
# 2.8.1. Kumpulkan semua kata unmatched dari dataframe hasil matching
from collections import Counter

all_unmatched = []
for lst in df['unmatched_words']: 
    all_unmatched.extend([w.lower() for w in lst])

# 2. Hitung frekuensi dan ambil top 50
unmatched_freq = Counter(all_unmatched)
top_50_unmatched = unmatched_freq.most_common(50)

# 3. Tampilkan
print("TOP 50 KATA UNMATCHED PALING SERING MUNCUL:")
print(f"{'Kata':<20} | {'Frekuensi':<10}")
print("-" * 35)
for word, freq in top_50_unmatched:
    print(f"{word:<20} | {freq:<10}")

TOP 50 KATA UNMATCHED PALING SERING MUNCUL:
Kata                 | Frekuensi 
-----------------------------------
dpr                  | 9948      
ruu                  | 6107      
ri                   | 5492      
rakyat               | 3159      
di                   | 3158      
dan                  | 3099      
ini                  | 1870      
reses                | 1737      
wakil                | 1675      
?                    | 1613      
untuk                | 1508      
dengan               | 1430      
ke                   | 1365      
pemerintah           | 1144      
ketua                | 1115      
tahun                | 1069      
masa                 | 939       
!                    | 932       
aspirasi             | 909       
jika                 | 764       
uu                   | 754       
fraksi               | 737       
ii                   | 703       
akan                 | 700       
pemilu               | 688       
oleh                 | 623       
pe

In [10]:
# 2.9 Preview Hasil Matching
print("\n[PREVIEW] 5 Tweet Pertama:")
for i in range(5):
    print(f"\nTweet {i+1}: {df['teks_processed'].iloc[i][:80]}...")
    print(f"  Matched (Total)      : {df['matched_words'].iloc[i][:5]}...")
    print(f"  Matched (Kata Fungsi): {df['matched_function_words'].iloc[i][:5]}...")
    print(f"  Unmatched            : {df['unmatched_words'].iloc[i][:5]}...")


[PREVIEW] 5 Tweet Pertama:

Tweet 1: ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
  Matched (Total)      : ['ADIL', 'yang', 'punya', 'kebijakan', 'ingat']...
  Matched (Kata Fungsi): ['yang', 'yang']...
  Unmatched            : ['loh', 'untuk', 'publik', 'negara', 'ini']...

Tweet 2: tertibkan media online DPR pemerintah jangan sporadis apalagi selektif hanya kep...
  Matched (Total)      : ['jangan', 'sporadis', 'selektif', 'hanya', 'yang']...
  Matched (Kata Fungsi): ['jangan', 'yang']...
  Unmatched            : ['tertibkan', 'media', 'online', 'DPR', 'pemerintah']...

Tweet 3: harus dievaluasi lagi kebijakan bebas visa terutama untuk negara tiongkok pak ! ...
  Matched (Total)      : ['harus', 'lagi', 'kebijakan', 'bebas', 'terutama']...
  Matched (Kata Fungsi): ['harus']...
  Unmatched            : ['dievaluasi', 'visa', 'untuk', 'negara', 'tiongkok']...

Tweet 4: jangan ngambang aturan logis apa undang undang...
  Matched (Total)      : ['jangan', 'apa

In [11]:
# 2.10 Simpan Output
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, 'lexicon_matching_tanpa_ignore.csv')
df.to_csv(output_path, index=False)

print(f"\n[OUTPUT] Data berhasil disimpan ke: {output_path}")
print("[CATATAN] Kolom 'matched_function_words' menyimpan kata fungsi yang terbawa masuk ke perhitungan skor.")


[OUTPUT] Data berhasil disimpan ke: ../../../outputs/sentiment-analysis/RSN\lexicon_matching_tanpa_ignore.csv
[CATATAN] Kolom 'matched_function_words' menyimpan kata fungsi yang terbawa masuk ke perhitungan skor.
